## How to Run in Google Colab

1.  **Clone the Repository:** Run the following command in a code cell to clone the QALF repository to your Colab environment.
2.  **Navigate to the Directory:** The notebook assumes it's run from within the `qalf` directory or its parent. After cloning, you might need to change your current working directory.
3.  **Install Dependencies:** If there are any specific Python package dependencies for `qalf` that are not pre-installed in Colab, you will need to install them (e.g., using `!pip install -e .` or `!pip install -r requirements.txt` if a `requirements.txt` is available in the repo).
4.  **Run Cells:** Execute the cells sequentially, starting from section 1.

In [1]:
import os

# Clone the repository
!git clone https://github.com/TheFausap/EXPLLM.git # Replace with the actual repository URL

# Change to the project directory if necessary
# The notebook's existing code tries to handle this, but you might need to adjust
# For example, if you want to be in the 'qalf' directory:
# %cd qalf

print(f"Current working directory: {os.getcwd()}")


Cloning into 'EXPLLM'...
remote: Enumerating objects: 250, done.
remote: Counting objects: 100% (250/250), done.
remote: Compressing objects: 100% (158/158), done.
remote: Total 250 (delta 148), reused 190 (delta 88), pack-reused 0 (from 0)
Receiving objects: 100% (250/250), 15.31 MiB | 12.63 MiB/s, done.
Resolving deltas: 100% (148/148), done.
Current working directory: /content


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---

# QALF Notebook: Quantum Associative Language Field

This notebook is kept as a thin, executable companion to the repository code. It imports the live `qalf` package instead of duplicating the tokenizer, data pipeline, model, and training loop inside notebook cells.

Current implementation snapshot:
- QALF-Mixed represents context as a mixture of complex Hilbert-space components.
- Decoding uses Born-style probabilities from complex amplitudes.
- Sparse bigram and trigram priors act as higher-order associative memory, not attention.
- Training supports job logs, resumable checkpoints, LR schedules, entropy regularisation, and prior decay.

## 1. Environment

Run from the repository root or from the `qalf/` directory. In Colab or a fresh machine, clone/copy the repository first, then set `ROOT` to the repository path.

In [ ]:
from pathlib import Path
import json
import sys
import torch

ROOT = Path.cwd()
if ROOT.name == "qalf":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from qalf.data import (
    build_tokenizer,
    encode_examples,
    make_windows,
    read_jsonl,
    relation_counts,
    trigram_counts,
)
from qalf.model import QALFConfig, QALFModel, cross_entropy_with_l2, device_for_training, load_checkpoint

print("root:", ROOT)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 2. Load Data And Build Associative Priors

The notebook defaults to `data/seed_corpus.jsonl` so it can run quickly. For bigger experiments, prepare TinyStories or another JSONL dataset with `qalf.prepare_text`, then point `DATA` to that file.

In [ ]:
DATA = "/content/drive/MyDrive/EXPLLM/data/dolly_qalf.jsonl"
examples = read_jsonl(DATA)
tokenizer = build_tokenizer(examples, vocab_size=512)
encoded = encode_examples(tokenizer, examples)

context_size = 24
contexts, prev2_tokens, prev_tokens, targets = make_windows(
    encoded,
    context_size=context_size,
    pad_id=tokenizer.pad_id,
    include_prev2=True,
)
bigram = relation_counts(encoded, len(tokenizer.vocab))
trigram = trigram_counts(encoded, len(tokenizer.vocab), top_k=16, min_count=1)

print("examples:", len(examples))
print("vocab:", len(tokenizer.vocab))
print("windows:", int(targets.numel()))
print("trigram contexts:", int(trigram["keys"].numel()))

## 3. Instantiate QALF-Mixed

A useful sanity check is `purity_mean`: with multiple context components it should usually be below 1.0, which means the context is being represented as a mixed density state rather than a single pure vector.

In [ ]:
device = device_for_training("auto")
config = QALFConfig(
    vocab_size=len(tokenizer.vocab),
    dimension=64,
    context_size=context_size,
    num_relations=4,
    num_components=4,
    bigram_strength=0.35,
    trigram_strength=0.75,
    pad_id=tokenizer.pad_id,
)
model = QALFModel(config, bigram_logits=bigram, trigram_prior=trigram).to(device)

batch = contexts[:8].to(device)
logits = model(batch, prev_tokens[:8].to(device), prev2_tokens[:8].to(device))
print("logits shape:", tuple(logits.shape))
print(model.diagnostics(batch))

## 4. Tiny Smoke Training

This is only a functionality check. It verifies forward/backward, the mixed-state diagnostics, and generation. Real experiments should use the CLI commands below so checkpointing and job logs are captured.

In [ ]:
dataset = torch.utils.data.TensorDataset(contexts, prev2_tokens, prev_tokens, targets)
loader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

model.train()
for step, (bc, bp2, bp, bt) in enumerate(loader, start=1):
    bc, bp2, bp, bt = bc.to(device), bp2.to(device), bp.to(device), bt.to(device)
    optimizer.zero_grad(set_to_none=True)
    logits = model(bc, bp, bp2)
    loss = cross_entropy_with_l2(model, logits, bt, entropy_weight=0.02)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    if step % 5 == 0 or step == 1:
        print({"step": step, "loss": float(loss.detach().cpu())})
    if step >= 20:
        break

model.eval()
with torch.no_grad():
    print(model.diagnostics(contexts[:8].to(device)))

In [ ]:
prompt = "What is QALF?"
print("prompt:", prompt)
print("reply:", model.generate(tokenizer, prompt, max_new_tokens=48, temperature=0.0, top_k=18, seed=7, device=device))

## 5. Prepare A Bigger Dataset

Example TinyStories preparation with a JSONL job log:

```bash
conda run -n EXPLLM python -m qalf.prepare_text \
  --source tinystories-train \
  --out data/tinystories_train_100k_qalf.jsonl \
  --max-examples 100000 \
  --prompt-tokens 24 \
  --reply-tokens 128 \
  --log-file runs/qalf_mixed_dgx/prep.jsonl
```

You can also pass a local file or JSONL dataset via `--source file` or `--source jsonl`.

## 6. Recommended QALF-Mixed Training Run

This is the current comparison-scale command for a larger GPU box. It uses constant LR because the first warmup-cosine run decayed too early for this underpowered objective. The CLI still supports `--lr-schedule warmup-cosine` for controlled tests.

```bash
conda run -n EXPLLM python -m qalf.train \
  --data data/tinystories_train_100k_qalf.jsonl \
  --out runs/qalf_mixed_compare \
  --device cuda \
  --dimension 512 \
  --context-size 128 \
  --components 6 \
  --vocab-size 32000 \
  --epochs 8 \
  --batch-size 2048 \
  --lr 0.004 \
  --lr-schedule constant \
  --max-windows 10000000 \
  --relations 12 \
  --trigram-top-k 96 \
  --trigram-min-count 2 \
  --trigram-strength 1.0 \
  --bigram-strength 0.35 \
  --entropy-weight 0.02 \
  --attractor-limit 2000 \
  --save-every 2 \
  --log-every 1 \
  --log-file runs/qalf_mixed_compare/train.jsonl
```

Useful extra flags:
- `--bigram-decay-epochs N` and `--trigram-decay-epochs N` to make the symbolic priors fade during training.
- `--bigram-device cpu` to save GPU memory if the vocabulary is large.
- `--resume runs/.../checkpoint_epoch_N.pt` to continue a run.
- `--reset-optimizer` when resuming across architecture changes.
- `--reset-lr-schedule` when resuming and intentionally restarting the LR schedule.

## 7. Resume, Evaluate, And Chat

Resume from a checkpoint:

```bash
conda run -n EXPLLM python -m qalf.train \
  --data data/tinystories_train_100k_qalf.jsonl \
  --out runs/qalf_mixed_compare \
  --resume runs/qalf_mixed_compare/checkpoint_epoch_8.pt \
  --epochs 16 \
  --device cuda \
  --batch-size 2048 \
  --lr 0.004 \
  --lr-schedule constant \
  --log-file runs/qalf_mixed_compare/resume.jsonl
```

Evaluate raw model quality without attractor retrieval:

```bash
conda run -n EXPLLM python -m qalf.eval \
  --checkpoint runs/qalf_mixed_compare/model.pt \
  --data data/tinystories_train_100k_qalf.jsonl \
  --out runs/qalf_mixed_compare/eval_raw.json \
  --device cuda \
  --no-attractor \
  --eval-batch-size 512 \
  --log-file runs/qalf_mixed_compare/eval_raw.jsonl
```

Chat with the trained checkpoint:

```bash
conda run -n EXPLLM python -m qalf.chat \
  --checkpoint runs/qalf_mixed_compare/model.pt \
  --device cuda \
  --prompt "Tell me a small story about a blue robot." \
  --max-new-tokens 80 \
  --temperature 0.8 \
  --top-k 24
```

## 8. Read Job Logs

Training and preparation logs are JSONL files. This helper prints the latest records from a run directory.

In [ ]:
LOG = ROOT / "runs" / "qalf_mixed_compare" / "train.jsonl"
if LOG.exists():
    lines = LOG.read_text(encoding="utf-8").strip().splitlines()
    for line in lines[-5:]:
        print(json.loads(line))
else:
    print("No log found yet:", LOG)

## 9. Notebook Drift Check

This notebook should stay small. If a future code change adds arguments or model fields, update the import/demo cells and the CLI recipes here, but keep the implementation in the Python package.